# Probe de clipping global de gradientes

Este notebook repite únicamente la seed 10 del piloto MLP de escala media y cambia una sola intervención de entrenamiento: clipping de norma global en `1.0`. No construye test.

Antes de ejecutar se congela el criterio para justificar una corrida de cinco seeds: el ratio de error final y el máximo tardío (épocas 10–20) deben ser como máximo `0.5`, el checkpoint debe conservar rango efectivo al menos `4`, la pureza debe ser al menos `49%` y el clipping debe haberse activado. La pureza mínima permite hasta dos puntos de caída frente al `51.07%` observado para esta seed sin clipping.

In [ ]:
# ruff: noqa: E402, E501
import json
import math
import random
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

from koopman_jepa.paper_config import (
    PaperCheckpointReplayConfig,
    load_paper_mlp_one_hidden_development_config,
)
from koopman_jepa.paper_data import PAPER_REGIME_NAMES, PaperRegimeDataset, generate_paper_master
from koopman_jepa.paper_evaluation import evaluate_paper_mlp_seed_clustering
from koopman_jepa.paper_model import PaperTemporalJEPA
from koopman_jepa.paper_training import (
    make_paper_loader,
    run_paper_train_validation_with_checkpoint,
    summarize_seed_checkpoint,
    verify_paper_checkpoint_replay,
)

plt.style.use("seaborn-v0_8-whitegrid")
torch.use_deterministic_algorithms(True)

In [ ]:
config = load_paper_mlp_one_hidden_development_config(
    ROOT / "configs" / "paper_mlp_gradient_clip_probe.yaml"
)
selection_gate = replace(config.checkpoint_gate, min_validation_effective_rank=1.0)
replay_config = PaperCheckpointReplayConfig(metric_absolute_tolerance=1e-8)
train_dataset = PaperRegimeDataset(config.data, "train", generate_paper_master)
validation_dataset = PaperRegimeDataset(config.data, "val", generate_paper_master)
train_keys = {train_dataset.sample_key(i) for i in range(len(train_dataset))}
validation_keys = {validation_dataset.sample_key(i) for i in range(len(validation_dataset))}
assert config.data.train_per_regime == 256
assert config.data.val_per_regime == 64
assert len(train_dataset) == 256 * len(PAPER_REGIME_NAMES) == 4608
assert len(validation_dataset) == 64 * len(PAPER_REGIME_NAMES) == 1152
assert train_keys.isdisjoint(validation_keys)
assert config.sweep.seeds == (10,)
assert config.train.max_gradient_norm == 1.0
print(json.dumps(asdict(config), indent=2))
print(f"Train/validation: {len(train_dataset)}/{len(validation_dataset)}. Test no fue instanciado.")

In [ ]:
seed = config.sweep.seeds[0]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
run_config = replace(config.train, seed=seed)
model = PaperTemporalJEPA(config.model)
started = time.perf_counter()
run = run_paper_train_validation_with_checkpoint(
    model, train_dataset, validation_dataset, run_config, selection_gate
)
elapsed = time.perf_counter() - started
assert run.selection is not None
replay = verify_paper_checkpoint_replay(
    model, run, validation_dataset, run_config, replay_config, expected_epoch=run.selection.epoch
)
assert replay.passed
summary = summarize_seed_checkpoint(seed, run.selection)
print(f"seed={seed} epoch={summary.checkpoint_epoch} val/base={summary.validation_loss_ratio:.4f} "
      f"gap={summary.validation_train_loss_ratio:.3f} rank={summary.validation_effective_rank:.2f} "
      f"time={elapsed:.1f}s")

In [ ]:
@torch.no_grad()
def collect_validation_embeddings():
    device = torch.device(run_config.device)
    model.to(device).eval()
    embeddings, labels = [], []
    for context, _, batch_labels in make_paper_loader(validation_dataset, run_config, shuffle=False):
        embeddings.append(model.online_encoder(context.to(device)).cpu().numpy())
        labels.append(batch_labels.numpy())
    return np.concatenate(embeddings), np.concatenate(labels)

embeddings, labels = collect_validation_embeddings()
clustering = evaluate_paper_mlp_seed_clustering(
    embeddings, labels, seed, config.clustering
)
print(json.dumps(asdict(clustering), indent=2))

In [ ]:
MAX_FINAL_VALIDATION_RATIO = 0.5
MAX_LATE_VALIDATION_RATIO = 0.5
MIN_CHECKPOINT_RANK = 4.0
MIN_MEAN_PURITY = 0.49
BASELINE_SEED10_PURITY = 0.5107
BASELINE_FINAL_VALIDATION_RATIO = 1014.3532

history = run.history
baseline_loss = history[0].validation_loss
epochs = np.array([row.epoch for row in history])
validation_ratios = np.array([row.validation_loss / baseline_loss for row in history])
online_gradients = np.array([row.online_gradient_norm for row in history])
predictor_gradients = np.array([row.predictor_gradient_norm for row in history])
combined_gradients = np.hypot(online_gradients, predictor_gradients)
final_validation_ratio = validation_ratios[-1]
max_late_validation_ratio = validation_ratios[epochs >= 10].max()
clip_activated = bool(np.any(combined_gradients[1:] > config.train.max_gradient_norm))
probe_passed = bool(
    final_validation_ratio <= MAX_FINAL_VALIDATION_RATIO
    and max_late_validation_ratio <= MAX_LATE_VALIDATION_RATIO
    and summary.validation_effective_rank >= MIN_CHECKPOINT_RANK
    and clustering.mean_purity >= MIN_MEAN_PURITY
    and clip_activated
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
axes[0].plot(epochs, validation_ratios, marker="o", markersize=3)
axes[0].axhline(MAX_FINAL_VALIDATION_RATIO, color="tab:red", linestyle="--", label="máximo")
axes[0].set_yscale("log")
axes[0].set(title="Error validation / inicial", xlabel="Época", ylabel="Ratio (log)")
axes[0].legend()
axes[1].plot(epochs[1:], online_gradients[1:], marker="o", markersize=3, label="encoder")
axes[1].plot(epochs[1:], predictor_gradients[1:], marker="o", markersize=3, label="predictor")
axes[1].axhline(config.train.max_gradient_norm, color="tab:red", linestyle="--", label="clip global")
axes[1].set_yscale("log")
axes[1].set(title="Normas pre-clipping", xlabel="Época", ylabel="Norma media (log)")
axes[1].legend()
axes[2].bar(["sin clip", "clip 1.0"], [BASELINE_SEED10_PURITY, clustering.mean_purity])
axes[2].axhline(MIN_MEAN_PURITY, color="tab:red", linestyle="--", label="mínimo probe")
axes[2].axhline(0.6548, color="tab:green", linestyle=":", label="paper")
axes[2].set(title="Pureza validation seed 10", ylabel="Pureza", ylim=(0.0, 0.75))
axes[2].legend()
plt.show()

display(Markdown(f"""## Análisis del resultado

- Resultado del probe: **{'PASS' if probe_passed else 'FAIL'}**.
- Ratio final: **{final_validation_ratio:.4f}** (sin clipping: `{BASELINE_FINAL_VALIDATION_RATIO:.1f}`).
- Máximo ratio entre épocas 10–20: **{max_late_validation_ratio:.4f}**.
- Rango efectivo del checkpoint: **{summary.validation_effective_rank:.2f}**.
- Pureza: **{clustering.mean_purity:.2%} ± {clustering.purity_std:.2%}** (sin clipping: `{BASELINE_SEED10_PURITY:.2%}`).
- El clipping se activó: **{clip_activated}**; máxima norma media pre-clipping: **{combined_gradients[1:].max():.2f}**.

Un PASS sólo habilita repetir esta condición en las cinco seeds de desarrollo. No reproduce todavía el `65.48%` del paper y no habilita test.
"""))
print("Test no fue construido ni consultado.")